# Phase 1: Data Understanding


## Step 01: Install Libraries


In [ ]:
!pip install kagglehub -q

## Step 02: Import Libraries


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

plt.style.use('ggplot')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries Loaded Successfully")

## Step 03: Download Dataset


In [ ]:
import kagglehub

path = kagglehub.dataset_download(
    "shivasingh4945/ai-impact-on-jobs-and-layoff-risk-dataset"
)

print("Dataset Path:")
print(path)

## Step 04: Locate Dataset Files


In [ ]:
import os

for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))

## Step 05: Load Dataset


In [ ]:
import os

files = os.listdir(path)

print(files)

df = pd.read_csv(os.path.join(path, files[0]))

df.head()

## Step 06: Dataset Shape


In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nShape:")
print(df.shape)

## Step 07: View First Records


In [ ]:
df.head(10)

## Step 08: View Last Records


In [ ]:
df.tail(10)

## Step 09: Column Names


In [ ]:
print("Columns:\n")

for col in df.columns:
    print(col)

## Step 10: Data Types


In [ ]:
df.dtypes

## Step 11: Dataset Information


In [ ]:
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum()/len(df))*100
})

missing.sort_values(
    by='Missing %',
    ascending=False
)

## Step 12: Duplicate Records


In [ ]:
duplicates = df.duplicated().sum()

print("Duplicate Rows:", duplicates)

## Step 13: Numerical Summary


In [ ]:
df.describe().T

## Step 14: Categorical Summary


In [ ]:
df.describe(include='object').T

## Step 15: Unique Values per Column


In [ ]:
unique_df = pd.DataFrame({
    "Column": df.columns,
    "Unique Values": [df[col].nunique() for col in df.columns]
})

unique_df.sort_values(
    by="Unique Values",
    ascending=False
)

## Step 16: Detect Numerical and Categorical Features


In [ ]:
numerical_cols = df.select_dtypes(
    include=['int64','float64']
).columns.tolist()

categorical_cols = df.select_dtypes(
    include=['object']
).columns.tolist()

print("Numerical Features:", len(numerical_cols))
print(numerical_cols)

print("\nCategorical Features:", len(categorical_cols))
print(categorical_cols)

## Step 17: Distribution of Numerical Features


In [ ]:
df[numerical_cols].hist(
    figsize=(20,15),
    bins=30
)

plt.tight_layout()
plt.show()

## Step 18: Boxplots for Outlier Detection


In [ ]:
for col in numerical_cols:

    plt.figure(figsize=(10,4))

    sns.boxplot(
        x=df[col]
    )

    plt.title(f"Boxplot of {col}")

    plt.show()

## Step 19: Correlation Matrix


In [ ]:
corr = df[numerical_cols].corr()

plt.figure(figsize=(15,10))

sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='coolwarm'
)

plt.title("Correlation Matrix")

plt.show()

## Step 20: Top Categories for Each Categorical Feature


In [ ]:
for col in categorical_cols:

    print("\n")
    print("="*60)
    print(col)
    print("="*60)

    print(df[col].value_counts().head(10))

## Step 21: Categorical Feature Visualization


In [ ]:
for col in categorical_cols:

    plt.figure(figsize=(12,5))

    df[col].value_counts().head(10).plot(
        kind='bar'
    )

    plt.title(col)

    plt.xticks(rotation=45)

    plt.show()

## Step 22: Identify Target Column


In [ ]:
for col in df.columns:

    print(col)

    if df[col].nunique() < 20:
        print(df[col].value_counts())

    print("-"*50)

## Step 23: Layoff Risk Distribution


In [ ]:
target = "Layoff_Risk"

plt.figure(figsize=(10,5))

sns.histplot(
    df[target],
    bins=30,
    kde=True
)

plt.title("Layoff Risk Distribution")

plt.show()

## Step 24: Feature Correlation With Target


In [ ]:
target = "Layoff_Risk"

# Map categorical 'Layoff_Risk' to numerical values for correlation
layoff_risk_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
df['Layoff_Risk_Encoded'] = df[target].map(layoff_risk_mapping)

# Calculate correlation with the new encoded target, including it temporarily in numerical_cols for the correlation matrix
corr_target = (
    df[numerical_cols + ['Layoff_Risk_Encoded']]
    .corr()['Layoff_Risk_Encoded']
    .drop('Layoff_Risk_Encoded') # Drop the target itself from the result
    .sort_values(ascending=False)
)

corr_target

## Step 25: Data Understanding Report


In [ ]:
print("="*60)
print("DATA UNDERSTANDING REPORT")
print("="*60)

print(f"Total Records : {df.shape[0]}")
print(f"Total Features: {df.shape[1]}")

print(f"\nNumerical Features : {len(numerical_cols)}")
print(f"Categorical Features : {len(categorical_cols)}")

print(f"\nDuplicate Rows : {df.duplicated().sum()}")

print("\nMissing Values:")
print(df.isnull().sum().sum())

print("\nDataset Ready For:")
print("✔ Data Cleaning")
print("✔ Feature Engineering")
print("✔ Machine Learning")
print("✔ Deep Learning")

# Phase 2: Data Preprocessing and Feature Engineering


## Step 01: Create Working Copy


In [ ]:
df_processed = df.copy()

print(df_processed.shape)

## Step 02: Check Missing Values Again


In [ ]:
missing = pd.DataFrame({
    'Missing_Count': df_processed.isnull().sum(),
    'Missing_Percentage':
        (df_processed.isnull().sum()/len(df_processed))*100
})

missing.sort_values(
    by='Missing_Percentage',
    ascending=False
)

## Step 03: Handle Missing Values


In [ ]:
num_cols = df_processed.select_dtypes(
    include=['int64','float64']
).columns

for col in num_cols:
    df_processed[col].fillna(
        df_processed[col].median(),
        inplace=True
    )

In [ ]:
cat_cols = df_processed.select_dtypes(
    include=['object']
).columns

for col in cat_cols:
    df_processed[col].fillna(
        df_processed[col].mode()[0],
        inplace=True
    )

## Step 04: Verify Missing Values


In [ ]:
df_processed.isnull().sum().sum()

## Step 05: Remove Duplicates


In [ ]:
before = df_processed.shape[0]

df_processed.drop_duplicates(
    inplace=True
)

after = df_processed.shape[0]

print("Removed:", before-after)

## Step 06: Encode Target Variable


In [ ]:
risk_mapping = {
    'Low':0,
    'Medium':1,
    'High':2
}

df_processed['Layoff_Risk'] = (
    df_processed['Layoff_Risk']
    .map(risk_mapping)
)

## Step 07: Verify Target


In [ ]:
df_processed['Layoff_Risk'].value_counts()

## Step 08: Detect Feature Types


In [ ]:
numerical_cols = df_processed.select_dtypes(
    include=['int64','float64']
).columns.tolist()

categorical_cols = df_processed.select_dtypes(
    include=['object']
).columns.tolist()

print("Numerical:", len(numerical_cols))
print("Categorical:", len(categorical_cols))

## Step 09: Remove IDs


In [ ]:
df_processed.columns.tolist()

In [ ]:
drop_cols = []

for col in df_processed.columns:

    if 'id' in col.lower():
        drop_cols.append(col)

print(drop_cols)

df_processed.drop(
    columns=drop_cols,
    inplace=True,
    errors='ignore'
)

## Step 10: One-Hot Encode Categories


In [ ]:
df_processed = pd.get_dummies(
    df_processed,
    drop_first=True
)

df_processed.head()

## Step 11: Verify Dataset


In [ ]:
print(df_processed.shape)

df_processed.head()

## Step 12: Correlation With Target


In [ ]:
corr_target = (
    df_processed
    .corr()['Layoff_Risk']
    .sort_values(ascending=False)
)

corr_target.head(20)

## Step 13: Feature Importance Heatmap


In [ ]:
plt.figure(figsize=(12,10))

sns.heatmap(
    df_processed.corr(),
    cmap='coolwarm',
    center=0
)

plt.title(
    "Feature Correlation Matrix"
)

plt.show()

## Step 14: Separate Features and Target


In [ ]:
X = df_processed.drop(
    ['Layoff_Risk', 'Layoff_Risk_Encoded'],
    axis=1
)

y = df_processed['Layoff_Risk']

print(X.shape)
print(y.shape)

## Step 15: Check Class Distribution


In [ ]:
y.value_counts()

## Step 16: Visualize Class Balance


In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(x=y)

plt.title(
    "Layoff Risk Distribution"
)

plt.show()

## Step 17: Handle Class Imbalance (SMOTE)


In [ ]:
!pip install imbalanced-learn -q

## Step 18: Apply SMOTE


In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    random_state=42
)

X_smote, y_smote = smote.fit_resample(
    X,
    y
)

print(X_smote.shape)
print(y_smote.shape)

## Step 19: Verify Balance


In [ ]:
pd.Series(
    y_smote
).value_counts()

## Step 20: Train-Test Split


In [ ]:
from sklearn.model_selection import (
    train_test_split
)

X_train, X_test, y_train, y_test = (
    train_test_split(
        X_smote,
        y_smote,
        test_size=0.2,
        random_state=42,
        stratify=y_smote
    )
)

## Step 21: Standardization


In [ ]:
from sklearn.preprocessing import (
    StandardScaler
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

## Step 22: Verify Shapes


In [ ]:
print("X_train:", X_train_scaled.shape)
print("X_test :", X_test_scaled.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

## Step 23: Save Processed Data


In [ ]:
import joblib

joblib.dump(
    scaler,
    "scaler.pkl"
)

df_processed.to_csv(
    "processed_layoff_dataset.csv",
    index=False
)

# Phase 3: Machine Learning Models


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import pandas as pd
import numpy as np

## Step 01: Evaluation Function


In [ ]:
from sklearn.ensemble import VotingClassifier
import numpy as np

def evaluate_model(model, X_test, y_test):
    if isinstance(model, VotingClassifier) and model.voting == 'hard':
        individual_predictions = [
            np.asarray(estimator.predict(X_test)).reshape(-1)
            for estimator in model.named_estimators_.values()
        ]

        encoded_predictions = np.vstack([
            model.le_.transform(pred)
            for pred in individual_predictions
        ]).T

        weights = getattr(model, '_weights_not_none', None)
        majority_votes = np.apply_along_axis(
            lambda row: np.argmax(
                np.bincount(
                    row.astype(int),
                    weights=weights,
                    minlength=len(model.le_.classes_)
                )
            ),
            axis=1,
            arr=encoded_predictions
        )
        y_pred = model.le_.inverse_transform(majority_votes)
    else:
        y_pred = np.asarray(model.predict(X_test)).reshape(-1)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    print('=' * 50)
    print(model.__class__.__name__)
    print('=' * 50)
    print('Accuracy :', accuracy)
    print('Precision:', precision)
    print('Recall   :', recall)
    print('F1 Score :', f1)
    print('\nClassification Report\n')
    print(classification_report(y_test, y_pred))

    return [
        model.__class__.__name__,
        accuracy,
        precision,
        recall,
        f1
    ]


## Step 02: Results Storage


In [ ]:
results = []

## Step 03: Model 1: Logistic Regression


In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr.fit(
    X_train_scaled,
    y_train
)

In [ ]:
results.append(
    evaluate_model(
        lr,
        X_test_scaled,
        y_test
    )
)

## Step 04: Model 2: Decision Tree


In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    random_state=42,
    max_depth=10
)

dt.fit(
    X_train,
    y_train
)

In [ ]:
results.append(
    evaluate_model(
        dt,
        X_test,
        y_test
    )
)

## Step 05: Model 3: Random Forest


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(
    X_train,
    y_train
)

In [ ]:
results.append(
    evaluate_model(
        rf,
        X_test,
        y_test
    )
)

## Step 06: Model 4: XGBoost


In [ ]:
!pip install xgboost -q

In [ ]:
from xgboost import XGBClassifier

num_classes = y_train.nunique() if hasattr(y_train, 'nunique') else len(np.unique(y_train))

xgb = XGBClassifier(
    objective='multi:softmax',
    num_class=num_classes,
    random_state=42,
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6
)

xgb.fit(
    X_train,
    y_train
)


In [ ]:
results.append(
    evaluate_model(
        xgb,
        X_test,
        y_test
    )
)

## Step 07: Model 5: LightGBM


In [ ]:
!pip install lightgbm -q

In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    random_state=42,
    n_estimators=300
)

lgbm.fit(
    X_train,
    y_train
)

In [ ]:
results.append(
    evaluate_model(
        lgbm,
        X_test,
        y_test
    )
)

## Step 08: Model 6: CatBoost


In [ ]:
!pip install catboost -q

In [ ]:
from catboost import CatBoostClassifier

cat = CatBoostClassifier(
    iterations=300,
    random_state=42,
    verbose=0
)

cat.fit(
    X_train,
    y_train
)

In [ ]:
results.append(
    evaluate_model(
        cat,
        X_test,
        y_test
    )
)

## Step 09: Model 7: Voting Ensemble


In [ ]:
from sklearn.ensemble import VotingClassifier

ensemble = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('xgb', xgb),
        ('lgbm', lgbm),
        ('cat', cat)
    ],
    voting='hard'
)

ensemble.fit(
    X_train,
    y_train
)

results.append(
    evaluate_model(
        ensemble,
        X_test,
        y_test
    )
)


## Step 10: Compare Models


In [ ]:
results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1"
    ]
)

results_df.sort_values(
    by="Accuracy",
    ascending=False
)

## Step 11: Performance Chart


In [ ]:
import matplotlib.pyplot as plt

results_df.sort_values(
    by='Accuracy',
    ascending=False
).plot(
    x='Model',
    y=['Accuracy'],
    kind='bar',
    figsize=(10,5)
)

plt.title(
    "Model Accuracy Comparison"
)

plt.show()

## Step 12: Feature Importance Analysis


In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

feature_importance.head(20)

In [ ]:
plt.figure(figsize=(10,8))

top_features = feature_importance.head(15)

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.title(
    "Top 15 Important Features"
)

plt.gca().invert_yaxis()

plt.show()

## Step 13: Confusion Matrix for Best Model


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

pred_rf = rf.predict(X_test)

cm = confusion_matrix(
    y_test,
    pred_rf
)

plt.figure(figsize=(7,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d'
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.title(
    "Random Forest Confusion Matrix"
)

plt.show()

## Step 14: Save Best Model


In [ ]:
import joblib

joblib.dump(
    rf,
    "best_layoff_model.pkl"
)

print("Model Saved")

# Phase 4: Deep Learning Models


## Step 01: Import Libraries


In [ ]:
import tensorflow as tf

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Dense,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

print(tf.__version__)


## Step 02: Verify Dataset Shapes


In [ ]:
print(X_train_scaled.shape)
print(X_test_scaled.shape)

print(y_train.shape)
print(y_test.shape)


## Step 03: Number of Features


In [ ]:
n_features = X_train_scaled.shape[1]

print("Features:", n_features)


## Step 04: Model 1: Basic ANN


In [ ]:
ann = Sequential([

    Dense(
        128,
        activation='relu',
        input_shape=(n_features,)
    ),

    Dropout(0.3),

    Dense(
        64,
        activation='relu'
    ),

    Dropout(0.2),

    Dense(
        32,
        activation='relu'
    ),

    Dense(
        3,
        activation='softmax'
    )
])

ann.summary()


## Step 05: Compile


In [ ]:
ann.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


## Step 06: Callbacks


In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    verbose=1
)


## Step 07: Train ANN


In [ ]:
history_ann = ann.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[
        early_stop,
        reduce_lr
    ],
    verbose=1
)


## Step 08: Evaluate ANN


In [ ]:
ann_loss, ann_acc = ann.evaluate(
    X_test_scaled,
    y_test,
    verbose=0
)

print("ANN Accuracy:", ann_acc)


## Step 09: Predictions


In [ ]:
import numpy as np

y_pred_ann = np.argmax(
    ann.predict(X_test_scaled),
    axis=1
)


## Step 10: Classification Report


In [ ]:
from sklearn.metrics import (
    classification_report
)

print(
    classification_report(
        y_test,
        y_pred_ann
    )
)


## Step 11: Confusion Matrix


In [ ]:
from sklearn.metrics import (
    confusion_matrix
)

import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(
    y_test,
    y_pred_ann
)

plt.figure(figsize=(7,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d'
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.title("ANN Confusion Matrix")

plt.show()


## Step 12: Model 2: Deep Neural Network (DNN)


In [ ]:
dnn = Sequential([

    Dense(
        256,
        activation='relu',
        input_shape=(n_features,)
    ),

    BatchNormalization(),

    Dropout(0.4),

    Dense(
        128,
        activation='relu'
    ),

    BatchNormalization(),

    Dropout(0.3),

    Dense(
        64,
        activation='relu'
    ),

    Dropout(0.2),

    Dense(
        32,
        activation='relu'
    ),

    Dense(
        3,
        activation='softmax'
    )
])

dnn.summary()


## Step 13: Compile


In [ ]:
dnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


## Step 14: Train DNN


In [ ]:
history_dnn = dnn.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=32,
    callbacks=[
        early_stop,
        reduce_lr
    ],
    verbose=1
)


## Step 15: Evaluate DNN


In [ ]:
dnn_loss, dnn_acc = dnn.evaluate(
    X_test_scaled,
    y_test,
    verbose=0
)

print("DNN Accuracy:", dnn_acc)


## Step 16: DNN Predictions


In [ ]:
y_pred_dnn = np.argmax(
    dnn.predict(X_test_scaled),
    axis=1
)


## Step 17: Classification Report


In [ ]:
print(
    classification_report(
        y_test,
        y_pred_dnn
    )
)


## Step 18: DNN Confusion Matrix


In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred_dnn
)

plt.figure(figsize=(7,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d'
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.title("DNN Confusion Matrix")

plt.show()


## Step 19: Learning Curves


In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    history_ann.history['accuracy'],
    label='Train Accuracy'
)

plt.plot(
    history_ann.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.title("ANN Learning Curve")

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend()

plt.show()


## Step 20: DNN Learning Curve


In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    history_dnn.history['accuracy'],
    label='Train Accuracy'
)

plt.plot(
    history_dnn.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.title("DNN Learning Curve")

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend()

plt.show()


## Step 21: Loss Curves


In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    history_dnn.history['loss'],
    label='Train Loss'
)

plt.plot(
    history_dnn.history['val_loss'],
    label='Validation Loss'
)

plt.legend()

plt.title("DNN Loss Curve")

plt.show()


## Step 22: Save Models


In [ ]:
ann.save(
    "ann_layoff_model.keras"
)

dnn.save(
    "dnn_layoff_model.keras"
)

print("Models Saved")


## Step 23: Compare Machine Learning vs Deep Learning


In [ ]:
comparison = pd.DataFrame({
    'Model':[
        'Logistic Regression',
        'Decision Tree',
        'Random Forest',
        'XGBoost',
        'LightGBM',
        'CatBoost',
        'ANN',
        'DNN'
    ],
    'Accuracy':[
        0.89,
        0.91,
        0.96,
        0.97,
        0.97,
        0.98,
        ann_acc,
        dnn_acc
    ]
})

comparison.sort_values(
    by='Accuracy',
    ascending=False
)


# Phase 5: Explainable AI


## Step 01: Install Libraries


In [ ]:
import importlib.util
import subprocess
import sys

required_packages = [
    'shap',
    'lime'
]

missing_packages = [
    package
    for package in required_packages
    if importlib.util.find_spec(package) is None
]

if missing_packages:
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        *missing_packages
    ])
else:
    print('SHAP and LIME are already installed.')


## Step 02: Imports


In [ ]:
import shap
import lime
import lime.lime_tabular

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt


## Step 03: Verify Best Model


In [ ]:
best_xai_model = cat
best_xai_model_name = "CatBoost"

print("Best XAI model:", best_xai_model_name)
best_xai_model


## Step 04: Create SHAP Explainer


In [ ]:
explainer = shap.TreeExplainer(best_xai_model)


## Step 05: Calculate SHAP Values


In [ ]:
sample_size = min(100, len(X_test))

sample_X = X_test.sample(
    sample_size,
    random_state=42
)

print("SHAP sample size:", sample_X.shape)

shap_values = explainer.shap_values(
    sample_X
)


## Step 06: Inspect Shape


In [ ]:
print(type(shap_values))

if isinstance(shap_values, list):
    print("Number of class arrays:", len(shap_values))
    print("Class 0 SHAP shape:", shap_values[0].shape)
else:
    print("SHAP shape:", np.asarray(shap_values).shape)


## Step 07: Prepare SHAP Values for Plots and Tables


In [ ]:
target_class_index = 2

if isinstance(shap_values, list):
    shap_values_for_class = shap_values[target_class_index]
    shap_importance_values = np.mean(
        [np.abs(class_values) for class_values in shap_values],
        axis=0
    )
elif np.asarray(shap_values).ndim == 3:
    shap_values_for_class = shap_values[:, :, target_class_index]
    shap_importance_values = np.abs(shap_values).mean(axis=2)
else:
    shap_values_for_class = shap_values
    shap_importance_values = np.abs(shap_values)

print("Selected class SHAP shape:", np.asarray(shap_values_for_class).shape)
print("Importance matrix shape:", np.asarray(shap_importance_values).shape)


## Step 08: Global Feature Importance


In [ ]:
shap.summary_plot(
    shap_values_for_class,
    sample_X,
    show=False
)

plt.tight_layout()
plt.show()
plt.close()


## Step 09: Bar Summary Plot


In [ ]:
shap.summary_plot(
    shap_values_for_class,
    sample_X,
    plot_type="bar",
    show=False
)

plt.tight_layout()
plt.show()
plt.close()


## Step 10: Feature Ranking Table


In [ ]:
importance = np.abs(
    shap_importance_values
).mean(axis=0)

importance_df = pd.DataFrame({
    'Feature': sample_X.columns,
    'Importance': importance
})

importance_df.sort_values(
    by='Importance',
    ascending=False
).head(20)


## Step 11: Dependence Plot


In [ ]:
shap.dependence_plot(
    "Tasks_Automated_Percentage",
    shap_values_for_class,
    sample_X,
    show=False
)

plt.tight_layout()
plt.show()
plt.close()


## Step 12: Another Dependence Plot


In [ ]:
shap.dependence_plot(
    "Routine_Task_Percentage",
    shap_values_for_class,
    sample_X,
    show=False
)

plt.tight_layout()
plt.show()
plt.close()
